# 전체 파이프라인 — 원본 csv 에서 제출 파일까지

이 노트북 하나가 **모든 것을 새로 만든다.** fold, 피처 파켓 22개, 모델 학습, 앙상블,
짝 규칙, 제출 파일, 실행 로그까지.

로직은 노트북에 복사하지 않는다. 전부 `scripts/` 와 `src/cancer_hack/` 을 불러 쓴다 —
CLI 와 노트북이 다른 코드를 지나면 노트북이 제출본을 재현하지 못하는 순간이 온다.

## 기존 산출물은 건드리지 않는다

`data/process/` 의 파켓과 `artifacts/` 의 예측 100여 개는 **지금까지 낸 제출본의 근거**다.
이 노트북은 `use_run_dirs()` 로 출력 위치를 실행별로 갈라 둔다.

```
data/process_<RUN_TAG>/          이번에 만든 피처 파켓
artifacts/runs/<RUN_TAG>/        이번 실행의 oof · test · log · submission
  ├── run.json                   설정·환경·지문·점수·소요 시간 전부
  └── run.md                     사람이 읽는 요약
```

원본 `data/raw/*.csv` 는 읽기만 한다.

**재검증할 때만** `reset_run_dirs()` 로 기존 경로로 돌아간다 — 마지막 절 참고.

## 왜 새로 만들어야 하나

`features_basic.encode_mutation` 이 바뀌었다. `*931*` 같은 동의 정지코돈을 2(기능 변이)가
아니라 1(동의 변이)로 센다. 정정이 맞지만, `data/process/mutation_encoded.parquet` 은
그 전 코드로 만들어졌다. `enc3`·`comut`·`lsvd`·`lnmf`·`gmod` 다섯 블록이 이 파일 하나에서
나오므로 **지금 코드로 다시 만들면 예전 OOF 와 비교가 끊긴다.** 그래서 새 폴더에 만든다.

## 실행 시간

GPU 기준 대략 이렇다. 피처 생성이 한 번 끝나면 `REBUILD_FEATURES = False` 로 두고
모델만 다시 돌릴 수 있다.

| 단계 | 시간 |
|---|---|
| fold + 피처 파켓 22개 | 15~25분 |
| 모델 3종 학습 (seed 1개) | 10분 |
| 앙상블 · 짝 규칙 · 제출 | 1분 미만 |

## 규정

- 인코더·스케일러·집계 통계는 전부 fold 의 train 부분에서만 fit 한다.
- 짝 규칙은 `train.csv` 에서만 유도되고 적용에는 test 한 행이면 된다.
- 외부 데이터 없음.
- **DACON 업로드는 사람이 직접 한다.** 이 노트북은 로컬 csv 만 만든다.

## 0. 설정

In [1]:
from __future__ import annotations

import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "scripts"))

# ---- 이번 실행 ---------------------------------------------------------------
RUN_TAG = "nb11_s42"           # 출력 폴더 이름이 된다. 공백·경로문자 금지
SEED = 42                      # 모델 시드
FOLD_SEED = 42                 # fold 분할 시드 (모델 시드와 별개)
N_SPLITS = 5
CV = "sgkf"                    # fold_group5 — 같은 변이 프로파일을 한 fold 로
CONFIG = "f16"                 # 피처셋 (train_gbdt.CONFIGS)
TOPK = 500
MODEL_WEIGHTS = {"xgb": 0.45, "catboost": 0.45, "rf": 0.10}
MIN_MUT = 3                    # 짝 규칙 최소 변이 수

REBUILD_FOLDS = True
REBUILD_FEATURES = True        # 한 번 만들었으면 False 로 두고 모델만 다시 돌린다
TRAIN_MODELS = True
# -----------------------------------------------------------------------------

from cancer_hack.paths import use_run_dirs
from cancer_hack.runlog import RunLog

DIRS = use_run_dirs(RUN_TAG)
RAW, PROC, ART = DIRS["raw"], DIRS["process"], DIRS["artifacts"]

for name in ("train.csv", "test.csv", "sample_submission.csv"):
    assert (RAW / name).exists(), f"{name} 이 {RAW} 에 없다"

LOG = RunLog(ART, run_tag=RUN_TAG, config={
    "seed": SEED, "fold_seed": FOLD_SEED, "n_splits": N_SPLITS, "cv": CV,
    "config": CONFIG, "topk": TOPK, "weights": MODEL_WEIGHTS, "min_mut": MIN_MUT,
})

print(f"원본   : {RAW}      (읽기만)")
print(f"피처   : {PROC}")
print(f"산출물 : {ART}")

원본   : D:\Code\Final_Hachathon\code\data\raw      (읽기만)
피처   : D:\Code\Final_Hachathon\code\data\process_nb11_s42
산출물 : D:\Code\Final_Hachathon\code\artifacts\runs\nb11_s42


### 환경 확인

블렌딩할 OOF 를 다른 기계에서 뽑아 합칠 때 버전이 어긋나면, 각자의 CV 는 멀쩡해 보이는데
**합쳐 놓은 결과만 조용히 어긋난다.** 그래서 예측을 바꾸는 라이브러리는 고정한다.

In [2]:
PINNED = {"scikit-learn": "1.9.0", "xgboost": "3.3.0",
          "lightgbm": "4.7.0", "catboost": "1.2.10"}
mismatch = []
for pkg, want in PINNED.items():
    got = LOG.environment.get(pkg, "없음")
    mark = "OK " if got == want else "다름"
    if got != want:
        mismatch.append(pkg)
    print(f"  [{mark}] {pkg:14s} 기준 {want:8s} 현재 {got}")
print(f"  [   ] torch          {LOG.environment.get('torch')}   (DL 을 쓸 때만 필요)")
if mismatch:
    print(f"\n※ {mismatch} 버전이 다르면 아래 점수가 재현되지 않는다.")

  [OK ] scikit-learn   기준 1.9.0    현재 1.9.0
  [OK ] xgboost        기준 3.3.0    현재 3.3.0
  [OK ] lightgbm       기준 4.7.0    현재 4.7.0
  [OK ] catboost       기준 1.2.10   현재 1.2.10
  [   ] torch          2.11.0+cu128   (DL 을 쓸 때만 필요)


## 1. fold 생성

`fold_skf5`(StratifiedKFold)와 `fold_group5`(**같은 변이 프로파일을 한 fold 로 묶는**
StratifiedGroupKFold) 두 벌을 한 파일에 만든다. 지금 쓰는 건 `fold_group5` 다.

프로파일이 같은 행을 갈라 놓으면 valid 에 train 의 쌍둥이가 들어가 점수가 부푼다.

In [3]:
from cancer_hack.validation import build_fold_frame, fold_class_distribution, fold_column
from cancer_hack.io import save_parquet
from cancer_hack.provenance import fold_fingerprint

FOLDS_PATH = PROC / "train_folds.parquet"

if REBUILD_FOLDS or not FOLDS_PATH.exists():
    with LOG.step("fold 생성"):
        folds = build_fold_frame(
            RAW / "train.csv",
            label_column="SUBCLASS",
            n_splits=N_SPLITS,
            seed=FOLD_SEED,
            group_cache_path=PROC / "train_group_keys.parquet",
        )
        save_parquet(folds, FOLDS_PATH)
        LOG.artifact("folds", FOLDS_PATH)
else:
    print(f"이미 있어 건너뛴다: {FOLDS_PATH}")

folds = pd.read_parquet(FOLDS_PATH)
FOLD_COLUMN = fold_column(CV, N_SPLITS)
LOG.record("fold_fingerprint", fold_fingerprint(FOLDS_PATH))
LOG.record("n_groups", int(folds["group_key"].nunique()))

print(f"{len(folds):,}행 · 그룹 {folds['group_key'].nunique():,}개")
print(f"{FOLD_COLUMN} 크기 {folds[FOLD_COLUMN].value_counts().sort_index().tolist()}")
print(f"지문 {LOG.values['fold_fingerprint']}")

[fold 생성] 시작


[fold 생성] 완료 · 9.5초



6,201행 · 그룹 5,636개
fold_group5 크기 [1241, 1240, 1241, 1239, 1240]
지문 997d89a20595cc23


## 2. 피처 파켓 생성

`scripts/make_features.py` 를 그대로 부른다. 서브커맨드마다 train/test 를 **따로** 돌린다 —
행마다 독립 계산이라 train 통계가 test 로 새지 않는다.

fold 안에서 잡아야 하는 것(`hypermutated_flag`·TF-IDF 어휘·공변이 선택·잠재 기저·
클래스 서명)은 여기서 만들지 않는다. 학습 스크립트가 fold 의 train 부분에 fit 한다.

In [4]:
# f16 16블록이 읽는 소스 전부. (서브커맨드, 추가 인자) 순서대로 돈다.
FEATURE_COMMANDS = [
    ("domain", []),                              # domain
    ("sample", ["--include-cell-rollup"]),       # rollup · rollup16
    ("enc3", []),                                # enc3 · comut · lsvd · lnmf · gmod
    ("gene", ["--kind", "event_count"]),         # gec
    ("gene", ["--kind", "mutated"]),             # ebovr 계열
    ("gene-types", []),                          # gtype
    ("parsed", []),                              # parsed19
    ("burden-extra", []),                        # burden8
    ("amino", []),                               # aa9
    ("sigtokens", []),                           # sigtok
    ("tokens", []),                              # exacttok
    ("parsed-tokens", []),                       # ptok
]

def run_make_features(command: str, extra: list[str], split: str) -> None:
    """make_features.py 를 서브프로세스로 부른다. 환경변수가 그대로 상속돼 출력 위치가 따라간다."""
    argv = [sys.executable, str(ROOT / "scripts/make_features.py"),
            command, "--split", split, "--overwrite", *extra]
    done = subprocess.run(argv, capture_output=True, text=True, encoding="utf-8", cwd=ROOT)
    if done.returncode != 0:
        raise RuntimeError(f"{command} {split} 실패\n{done.stdout[-2000:]}\n{done.stderr[-2000:]}")

if REBUILD_FEATURES:
    with LOG.step("피처 파켓 생성"):
        for command, extra in FEATURE_COMMANDS:
            for split in ("train", "test"):
                label = f"{command}{''.join(' ' + e for e in extra)} [{split}]"
                run_make_features(command, extra, split)
                print(f"  {label}")
else:
    print("REBUILD_FEATURES=False — 기존 파켓을 그대로 쓴다")

made = sorted(p.name for p in PROC.glob("*.parquet"))
LOG.record("n_feature_parquets", len(made))
print(f"\n파켓 {len(made)}개")

[피처 파켓 생성] 시작


  domain [train]


  domain [test]


  sample --include-cell-rollup [train]


  sample --include-cell-rollup [test]


...

  tokens [train]


  tokens [test]


  parsed-tokens [train]


  parsed-tokens [test]
[피처 파켓 생성] 완료 · 944.4초




파켓 26개


### 새 파켓의 지문을 남긴다

나중에 "이 점수가 어느 파켓에서 나왔나"를 되짚는 유일한 단서다. 파일 바이트가 아니라
**내용** 지문이라 압축·행 순서가 바뀌어도 안 흔들린다.

In [5]:
from cancer_hack.provenance import parquet_fingerprint

with LOG.step("파켓 지문"):
    fingerprints = {p.name: parquet_fingerprint(p) for p in sorted(PROC.glob("*.parquet"))}
    LOG.record("feature_fingerprints", fingerprints)

for name, fp in list(fingerprints.items())[:6]:
    print(f"  {name:48s} {fp}")
print(f"  … 총 {len(fingerprints)}개")

[파켓 지문] 시작


[파켓 지문] 완료 · 4.6초



  test_additional_burden_features.parquet          ac2a455d64a21f5d
  test_amino_acid_features.parquet                 039cc2f7243b9884
  test_domain_features.parquet                     b942a552d7efd62e
  test_exact_mutation_tokens.parquet               9b659a77460d903e
  test_gene_event_count_matrix.parquet             9f4a0586adb1c3d8
  test_gene_mutated_matrix.parquet                 9bd61056b841ec9d
  … 총 26개


## 3. 모델 학습

`train_gbdt.run_config` 를 그대로 부른다 — CLI 와 **같은 코드 경로**다.
`Dataset` 을 한 번 만들어 세 모델이 나눠 쓴다.

In [6]:
from train_gbdt import CONFIGS, Dataset, build_parser as tg_parser, run_config

def train(model: str, data) -> dict:
    args = tg_parser().parse_args([])      # 피처 축 30여 개를 기본값으로 받는다
    args.model = model
    args.topk = TOPK
    args.n_splits = N_SPLITS
    args.seed = SEED
    args.device = "auto"
    args.override = {}
    args.params_preset = None              # 기본 하이퍼파라미터. 프리셋은 --params 로
    args.dry_run = False
    args.submission = False
    args.tag = RUN_TAG
    args.gpu_ram_part = 0.4
    return run_config(data, config=CONFIG, cv=CV, args=args)

RESULTS: dict[str, dict] = {}
if TRAIN_MODELS:
    with LOG.step("Dataset 준비"):
        DATA = Dataset(set(CONFIGS[CONFIG]["blocks"]), n_splits=N_SPLITS)
    for model in MODEL_WEIGHTS:
        with LOG.step(f"학습 {model}"):
            RESULTS[model] = train(model, DATA)
            LOG.record(f"oof_{model}", RESULTS[model]["oof_macro_f1"])
            print(f"    OOF macro F1 = {RESULTS[model]['oof_macro_f1']:.4f}")
else:
    print("TRAIN_MODELS=False — 이 실행 폴더의 기존 OOF 를 쓴다")

[Dataset 준비] 시작


[block] aa9            9열  아미노산 치환 페널티 9종


[block] burden8        8열  추가 burden 8종


[block] comut      4,384열  공변이 쌍 (fold 안 선택)


[block] csig       4,384열  클래스 서명 몫 (fold 안 라벨 선택)


...

  [rf_nb11_s42_f16_group5_k500_sp1000m3p2_cm20drishamm83824c_lt64svdnmfl257c23c_gm24shac918bc_sg30sha71bc36_s42] fold 4/5  dim=5,314  Macro F1=0.4672  (17s)


D:\Code\Final_Hachathon\code\.venv\Lib\site-packages\sklearn\decomposition\_nmf.py:1723: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(


  [rf_nb11_s42_f16_group5_k500_sp1000m3p2_cm20drishamm83824c_lt64svdnmfl257c23c_gm24shac918bc_sg30sha71bc36_s42] fold 5/5  dim=5,309  Macro F1=0.4810  (17s)


  [rf_nb11_s42_f16_group5_k500_sp1000m3p2_cm20drishamm83824c_lt64svdnmfl257c23c_gm24shac918bc_sg30sha71bc36_s42] OOF Macro F1 = 0.4761  단독행 = 0.4708  Acc = 0.4672  (83s)


    OOF macro F1 = 0.4761
[학습 rf] 완료 · 83.1초



## 4. 앙상블 — 고정 가중 + 로짓 보정

가중치 `0.45 / 0.45 / 0.10` 고정. 보정은 클래스별로 로짓에 상수를 더해 결정 경계를 옮긴다.
**교차적합으로 한다** — fold 를 뺀 나머지에서 바이어스를 찾고 그 fold 에만 적용한다.
전체 OOF 에 한 번에 맞추면 점수가 부풀어 구성을 고르는 근거로 못 쓴다.

`scripts/calibrate_ensemble.py` 가 부르는 것과 **같은 함수**를 쓴다.

In [7]:
from cancer_hack.calibration import MacroF1LogitBias
from cancer_hack.ensemble import crossfit_calibrated_blend, weighted_average
from cancer_hack.metrics import macro_f1, read_prediction_frame

def load(kind: str, model: str):
    stem = RESULTS[model]["stem"] if RESULTS else None
    if stem is None:
        hits = sorted((ART / kind).glob(f"{'oof' if kind == 'oof' else 'test'}_{model}_{RUN_TAG}_*.csv"))
        assert len(hits) == 1, f"{model} {kind} 후보가 {len(hits)}개다"
        path = hits[0]
    else:
        path = ART / kind / f"{'oof' if kind == 'oof' else 'test'}_{stem}.csv"
    frame, classes = read_prediction_frame(path)
    return frame.sort_values("ID", kind="stable"), classes

with LOG.step("앙상블"):
    oof_frames = [load("oof", m) for m in MODEL_WEIGHTS]
    test_frames = [load("test_predictions", m) for m in MODEL_WEIGHTS]
    classes = oof_frames[0][1]
    assert all(c == classes for _, c in oof_frames + test_frames), "클래스 구성이 다르다"

    columns = [f"p_{c}" for c in classes]
    oof = [f[columns].to_numpy(dtype=float) for f, _ in oof_frames]
    test = [f[columns].to_numpy(dtype=float) for f, _ in test_frames]
    y = oof_frames[0][0]["y_true"].to_numpy()

    ids = oof_frames[0][0]["ID"].astype(str).to_numpy()
    fold_ids = folds.set_index(folds["ID"].astype(str)).reindex(ids)[FOLD_COLUMN].to_numpy()
    weights = list(MODEL_WEIGHTS.values())

    raw_blend, crossfit, _ = crossfit_calibrated_blend(oof, y, classes, fold_ids, weights)
    LOG.record("oof_blend_raw", macro_f1(y, np.asarray(classes)[raw_blend.argmax(1)]))
    LOG.record("oof_blend_calibrated", macro_f1(y, np.asarray(classes)[crossfit.argmax(1)]))

    # test 에는 정답이 없어 교차적합을 못 한다. 전체 OOF 로 맞춘 바이어스를 쓴다.
    final_bias = MacroF1LogitBias().fit(raw_blend, y, classes)
    test_proba = final_bias.predict_proba(weighted_average(test, weights))

print(f"보정 전 blend OOF = {LOG.values['oof_blend_raw']:.4f}")
print(f"보정 후 blend OOF = {LOG.values['oof_blend_calibrated']:.4f}   ← 보고할 값")

[앙상블] 시작


[앙상블] 완료 · 16.8초



보정 전 blend OOF = 0.5004
보정 후 blend OOF = 0.5132   ← 보고할 값


## 5. 짝 라벨 규칙

test 행의 유전자 프로파일이 train 의 **유일한** 행과 바이트 단위로 같고 그 train 라벨이
`KIPAN`·`KIRC`·`GBMLGG`·`LGG` 중 하나면, 예측을 **짝 코호트 라벨**로 바꾼다.
매칭된 라벨이 아니라 반대쪽을 쓴다는 게 핵심이다.

근거·규정 판정·측정 불가 이유는 `docs/pair_rule.md`. 전제가 깨지면 아래 셀이 멈춘다.

In [8]:
from cancer_hack.pair_rule import PAIR, build_pair_rule

with LOG.step("짝 규칙"):
    rule = build_pair_rule(RAW / "train.csv", RAW / "test.csv", min_mut=MIN_MUT)
    violations = rule.verify_premises()
    assert violations == [], violations
    LOG.record("pair_rule_rows", rule.diagnostics["n_flipped"])
    LOG.record("pair_rule_diagnostics", rule.diagnostics)

d = rule.diagnostics
print(f"중복 묶음  같은 라벨 {d['train_dup_groups_same_label']} / 짝 라벨 {d['train_dup_groups_pair_label']}")
print(f"고아 수    {d['train_orphans_by_label']}")
print(f"test 매칭  {d['test_matched_by_train_label']}")
print(f"규칙 대상  {d['n_flipped']}행 · 전제 검사 통과")

[짝 규칙] 시작


[짝 규칙] 완료 · 0.4초



중복 묶음  같은 라벨 0 / 짝 라벨 422
고아 수    {'KIPAN': 233, 'GBMLGG': 280, 'KIRC': 57, 'LGG': 50}
test 매칭  {'LGG': 50, 'KIPAN': 59, 'KIRC': 57, 'GBMLGG': 48}
규칙 대상  214행 · 전제 검사 통과


## 6. 제출 파일

In [9]:
with LOG.step("제출 파일"):
    sample = pd.read_csv(RAW / "sample_submission.csv")
    test_ids = test_frames[0][0]["ID"].astype(str).to_numpy()
    base = pd.DataFrame({"ID": test_ids, "SUBCLASS": np.asarray(classes)[test_proba.argmax(1)]})
    base = sample[["ID"]].astype(str).merge(base, on="ID", validate="one_to_one")

    final = base.copy()
    before = base["SUBCLASS"].to_numpy().copy()
    final["SUBCLASS"] = rule.relabel(final["ID"], before)
    changed = int((final["SUBCLASS"].to_numpy() != before).sum())
    LOG.record("pair_rule_changed", changed)

    for name, frame in (("submission_base.csv", base), ("submission_pairrule.csv", final)):
        assert list(frame.columns) == ["ID", "SUBCLASS"] and len(frame) == 2546
        assert (frame["ID"].to_numpy() == sample["ID"].astype(str).to_numpy()).all()
        assert frame["SUBCLASS"].notna().all() and set(frame["SUBCLASS"]) <= set(classes)
        path = LOG.artifact(name.removesuffix(".csv"), ART / "submissions" / name)
        frame.to_csv(path, index=False, encoding="UTF-8-sig")

print(f"짝 규칙으로 바뀐 행 {changed}")
print(f"  {ART / 'submissions' / 'submission_base.csv'}")
print(f"  {ART / 'submissions' / 'submission_pairrule.csv'}")
print("\n로컬 파일만 만들었다. DACON 업로드는 사람이 직접 한다.")

[제출 파일] 시작


[제출 파일] 완료 · 0.0초



짝 규칙으로 바뀐 행 202
  D:\Code\Final_Hachathon\code\artifacts\runs\nb11_s42\submissions\submission_base.csv
  D:\Code\Final_Hachathon\code\artifacts\runs\nb11_s42\submissions\submission_pairrule.csv

로컬 파일만 만들었다. DACON 업로드는 사람이 직접 한다.


## 7. 실행 기록 저장

`run.json` 에 설정·환경·파켓 지문·단계별 시간·점수가 전부 들어간다. `run.md` 는 같은 내용의
사람이 읽는 요약이다. 이 둘만 있으면 나중에 이 실행을 그대로 되짚을 수 있다.

In [10]:
paths = LOG.save()
print(LOG.summary())
print()
for label, path in paths.items():
    print(f"  {label:9s} {path}")
print()
print(pd.DataFrame(
    [(s["name"], s.get("status"), s.get("seconds")) for s in LOG.steps],
    columns=["단계", "상태", "초"],
).to_string(index=False))

nb11_s42 · 단계 10개 · 25.8분
전 단계 완료

  json      D:\Code\Final_Hachathon\code\artifacts\runs\nb11_s42\run.json
  markdown  D:\Code\Final_Hachathon\code\artifacts\runs\nb11_s42\run.md

         단계 상태     초
    fold 생성 완료   9.5
   피처 파켓 생성 완료 944.4
      파켓 지문 완료   4.6
 Dataset 준비 완료   2.9
     학습 xgb 완료 252.4
학습 catboost 완료 234.1
      학습 rf 완료  83.1
        앙상블 완료  16.8
       짝 규칙 완료   0.4
      제출 파일 완료   0.0


## 8. 재검증 — 기존 산출물로 돌아가기

`data/process/` 와 `artifacts/` 는 지금까지 낸 제출본의 근거다. 그쪽 숫자를 다시 확인할
때만 경로를 되돌린다.

```python
from cancer_hack.paths import reset_run_dirs
reset_run_dirs()
```

되돌린 뒤에는 `notebooks/09_final_submission.ipynb`(LB 0.4818)와
`notebooks/10_seed_ensemble_submission.ipynb`(LB 0.4725)가 각자의 제출본을 전 행 대조한다.
`tests/test_data_provenance.py` 가 그 파켓들이 안 바뀌었는지 지킨다.

**이 노트북이 만든 파켓과 기존 파켓은 섞지 않는다.** `encode_mutation` 이 바뀌어
`enc3` 계열 값이 다르므로, 두 쪽 OOF 를 한 블렌드에 넣으면 안 된다.